# Enhanced Feature Engineering for Real Estate Data

This notebook demonstrates how to use the enhanced feature engineering capabilities to extract more meaningful features from real estate data.

## Table of Contents

1. [Setup](#Setup)
2. [Loading Sample Data](#Loading-Sample-Data)
3. [Basic Feature Engineering](#Basic-Feature-Engineering)
4. [Price Trend Analysis](#Price-Trend-Analysis)
5. [Neighborhood Clustering](#Neighborhood-Clustering)
6. [Sentiment Analysis](#Sentiment-Analysis)
7. [Energy Efficiency Features](#Energy-Efficiency-Features)
8. [External Data Enrichment](#External-Data-Enrichment)
9. [Combining Features](#Combining-Features)
10. [Feature Importance Analysis](#Feature-Importance-Analysis)

## Setup

First, let's import the necessary modules and configure the environment.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

# Configure paths and settings
sys.path.append('..')
from advanced_features import (
    DescriptionFeatureExtractor,
    GeographicFeatureTransformer,
    TimeFeatureTransformer,
    TextVectorizerTransformer,
    create_advanced_feature_pipeline
)

from enhanced_features import (
    PriceTrendAnalyzer,
    NeighborhoodClusterer,
    SentimentAnalysisTransformer,
    EnergyEfficiencyTransformer,
    ExternalDataEnricher,
    create_enhanced_feature_pipeline,
    feature_importance_analysis,
    calculate_property_age_features
)

from model_utils import ModelRegistry, HyperparameterOptimizer

# Set up plotting
%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set(style="whitegrid")

# Create output directories
os.makedirs('outputs/plots', exist_ok=True)
os.makedirs('outputs/models', exist_ok=True)
os.makedirs('outputs/data', exist_ok=True)

## Loading Sample Data

Let's load some sample real estate data to work with. In a real-world scenario, this would be your dataset of property listings.

In [ ]:
# Check if we have a sample dataset, otherwise create one
sample_data_path = 'outputs/data/sample_real_estate_data.csv'

if os.path.exists(sample_data_path):
    # Load existing sample data
    df = pd.read_csv(sample_data_path)
    print(f"Loaded sample dataset with {len(df)} properties")
else:
    # In a real scenario, you would load your actual data here
    # For demonstration, let's create a synthetic dataset
    print("Creating synthetic dataset for demonstration...")
    
    # Number of properties
    n_properties = 1000
    
    # Generate random data
    np.random.seed(42)
    
    # Cities and their approximate coordinates (latitude, longitude)
    cities = {
        'Milano': (45.4642, 9.1900),
        'Roma': (41.9028, 12.4964),
        'Napoli': (40.8518, 14.2681),
        'Torino': (45.0703, 7.6869),
        'Genova': (44.4056, 8.9463)
    }
    
    city_names = list(cities.keys())
    
    # Generate property data
    data = {
        'property_id': [f"PROP{i:06d}" for i in range(n_properties)],
        'city': np.random.choice(city_names, n_properties),
        'date_posted': pd.date_range(
            start=datetime.now() - timedelta(days=365), 
            end=datetime.now(), 
            periods=n_properties
        ),
        'surface_area': np.random.normal(100, 30, n_properties).round().clip(30, 300).astype(int),
        'rooms': np.random.choice([1, 2, 3, 4, 5], n_properties, p=[0.1, 0.3, 0.4, 0.15, 0.05]),
        'bathrooms': np.random.choice([1, 2, 3], n_properties, p=[0.5, 0.4, 0.1]),
        'energy_class': np.random.choice(['A', 'A+', 'B', 'C', 'D', 'E', 'F', 'G'], n_properties)
    }
    
    # Add coordinates based on city with some noise
    lat, lon = [], []
    for city in data['city']:
        base_lat, base_lon = cities[city]
        lat.append(base_lat + np.random.normal(0, 0.02))
        lon.append(base_lon + np.random.normal(0, 0.02))
    
    data['latitude'] = lat
    data['longitude'] = lon
    
    # Add price (function of city, area, rooms, energy class with noise)
    base_prices = {
        'Milano': 4000,
        'Roma': 3500,
        'Napoli': 2500,
        'Torino': 2000,
        'Genova': 2200
    }
    
    energy_multiplier = {
        'A': 1.2, 'A+': 1.25, 'B': 1.1, 'C': 1.0, 
        'D': 0.95, 'E': 0.9, 'F': 0.85, 'G': 0.8
    }
    
    prices = []
    for i in range(n_properties):
        city_price = base_prices[data['city'][i]]
        area = data['surface_area'][i]
        rooms = data['rooms'][i]
        energy = energy_multiplier[data['energy_class'][i]]
        
        # Base price calculation
        price = city_price * area * energy * (1 + 0.05 * rooms)
        
        # Add noise
        price *= np.random.normal(1, 0.15)
        
        prices.append(int(price / 1000) * 1000)  # Round to nearest thousand
    
    data['price'] = prices
    
    # Add price per sqm
    data['price_per_sqm'] = (data['price'] / data['surface_area']).round(2)
    
    # Generate synthetic descriptions
    descriptions = []
    for i in range(n_properties):
        city = data['city'][i]
        rooms = data['rooms'][i]
        area = data['surface_area'][i]
        energy = data['energy_class'][i]
        price = data['price'][i]
        
        # Random features
        features = []
        if np.random.random() > 0.5:
            features.append("balcone")
        if np.random.random() > 0.7:
            features.append("terrazzo")
        if np.random.random() > 0.6:
            features.append("posto auto")
        if np.random.random() > 0.8:
            features.append("aria condizionata")
            
        # Random year built
        year_built = np.random.randint(1950, 2022)
        
        # Random renovation status
        if year_built < 2000 and np.random.random() > 0.7:
            renovation_year = np.random.randint(year_built + 10, 2022)
            renovation_text = f" ristrutturato nel {renovation_year},"
        else:
            renovation_text = ""
        
        # Create description
        desc = f"""Appartamento di {area} mq in {city}, costruito nel {year_built}{renovation_text} 
                con {rooms} {'camera' if rooms == 1 else 'camere'} da letto e {data['bathrooms'][i]} {'bagno' if data['bathrooms'][i] == 1 else 'bagni'}. 
                Classe energetica {energy}. """
        
        # Add features
        if features:
            desc += "Dotato di " + ", ".join(features) + ". "
            
        # Add random qualitative descriptions
        qualities = [
            "Luminoso e spazioso",
            "In zona tranquilla",
            "Vicino ai mezzi pubblici",
            "Ben collegato al centro",
            "Recentemente ristrutturato",
            "In ottimo stato",
            "Finiture di pregio"
        ]
        
        selected_qualities = np.random.choice(qualities, size=np.random.randint(1, 4), replace=False)
        desc += " " + ". ".join(selected_qualities) + "."
        
        descriptions.append(desc)
    
    data['description'] = descriptions
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Save the sample data
    df.to_csv(sample_data_path, index=False)
    print(f"Created synthetic dataset with {len(df)} properties")

# Display sample data
df.head()

In [ ]:
# Basic statistics of the dataset
df.describe()

## Basic Feature Engineering

Let's start with the basic feature engineering using the existing transformers from `advanced_features.py`.

In [ ]:
# Extract description features
description_extractor = DescriptionFeatureExtractor()
desc_features = description_extractor.fit_transform(df)

print(f"Extracted {desc_features.shape[1]} features from descriptions")
desc_features.head()

In [ ]:
# Extract geographic features
geo_transformer = GeographicFeatureTransformer()
geo_features = geo_transformer.fit_transform(df)

print(f"Extracted {geo_features.shape[1]} geographic features")
geo_features.head()

In [ ]:
# Extract time-based features
time_transformer = TimeFeatureTransformer()
time_features = time_transformer.fit_transform(df)

print(f"Extracted {time_features.shape[1]} time-based features")
time_features.head()

## Price Trend Analysis

Now, let's use the enhanced feature engineering capabilities, starting with price trend analysis.

In [ ]:
# Extract price trend features
trend_analyzer = PriceTrendAnalyzer(window_sizes=[30, 90, 180])
trend_features = trend_analyzer.fit_transform(df)

print(f"Extracted {trend_features.shape[1]} price trend features")
trend_features.head()

In [ ]:
# Visualize price trends by city
plt.figure(figsize=(12, 6))

for city in df['city'].unique():
    city_data = df[df['city'] == city].sort_values('date_posted')
    plt.plot(city_data['date_posted'], city_data['price_per_sqm'].rolling(20).mean(), label=city)

plt.title('Price per Square Meter Trends by City')
plt.xlabel('Date')
plt.ylabel('Price per Square Meter (€)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/plots/price_trends.png', dpi=300)
plt.show()

## Neighborhood Clustering

Next, let's cluster the properties into neighborhoods based on their geographic location and price.

In [ ]:
# Perform neighborhood clustering
clusterer = NeighborhoodClusterer(n_clusters=8)
neighborhood_features = clusterer.fit_transform(df)

print(f"Extracted {neighborhood_features.shape[1]} neighborhood features")
neighborhood_features.head()

In [ ]:
# Visualize the neighborhood clusters
fig = clusterer.visualize_clusters(
    df, 
    output_path='outputs/plots/neighborhood_clusters.png',
    figsize=(14, 10)
)

In [ ]:
# Analyze neighborhood statistics
cluster_stats = pd.DataFrame([
    {
        'cluster': cluster_id,
        'count': stats['count'],
        'avg_price': stats.get('avg_price', 0),
        'avg_price_per_sqm': stats.get('avg_price_per_sqm', 0),
        'latitude': stats['centroid']['latitude'],
        'longitude': stats['centroid']['longitude']
    }
    for cluster_id, stats in clusterer.cluster_stats.items()
])

cluster_stats.sort_values('avg_price_per_sqm', ascending=False)

## Sentiment Analysis

Now, let's analyze the sentiment in property descriptions.

In [ ]:
# Extract sentiment features
sentiment_analyzer = SentimentAnalysisTransformer()
sentiment_features = sentiment_analyzer.fit_transform(df)

print(f"Extracted {sentiment_features.shape[1]} sentiment features")
sentiment_features.head()

In [ ]:
# Analyze relationship between sentiment and price
plt.figure(figsize=(10, 6))
plt.scatter(
    sentiment_features['sentiment_score'],
    df['price_per_sqm'],
    alpha=0.6,
    c=sentiment_features['positive_term_count'],
    cmap='viridis'
)
plt.colorbar(label='Positive Term Count')
plt.title('Relationship Between Sentiment Score and Price per Square Meter')
plt.xlabel('Sentiment Score')
plt.ylabel('Price per Square Meter (€)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/plots/sentiment_vs_price.png', dpi=300)
plt.show()

## Energy Efficiency Features

Let's extract features related to energy efficiency.

In [ ]:
# Extract energy efficiency features
energy_transformer = EnergyEfficiencyTransformer()
energy_features = energy_transformer.fit_transform(df)

print(f"Extracted {energy_features.shape[1]} energy efficiency features")
energy_features.head()

In [ ]:
# Analyze price premium for energy efficiency
if 'energy_class_value' in energy_features.columns:
    combined_data = pd.concat([df['price_per_sqm'], energy_features['energy_class_value']], axis=1)
    combined_data = combined_data[combined_data['energy_class_value'] >= 0]  # Filter out unknown/exempt
    
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='energy_class_value', y='price_per_sqm', data=combined_data)
    plt.title('Price per Square Meter by Energy Efficiency Class')
    plt.xlabel('Energy Class Value (0=G to 10=A4)')
    plt.ylabel('Price per Square Meter (€)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('outputs/plots/energy_vs_price.png', dpi=300)
    plt.show()

## Property Age Features

Let's extract and analyze property age features.

In [ ]:
# Extract property age features
age_features = calculate_property_age_features(df)

print(f"Extracted {age_features.shape[1]} property age features")
age_features.head()

In [ ]:
# Analyze relationship between property age and price
if 'property_age' in age_features.columns:
    valid_age = age_features['property_age'].notna()
    
    plt.figure(figsize=(10, 6))
    plt.scatter(
        age_features.loc[valid_age, 'property_age'],
        df.loc[valid_age, 'price_per_sqm'],
        alpha=0.6,
        c=df.loc[valid_age, 'surface_area'],
        cmap='coolwarm'
    )
    plt.colorbar(label='Surface Area (m²)')
    plt.title('Relationship Between Property Age and Price per Square Meter')
    plt.xlabel('Property Age (years)')
    plt.ylabel('Price per Square Meter (€)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('outputs/plots/age_vs_price.png', dpi=300)
    plt.show()

## Combining Features

Now, let's combine all these features into a comprehensive feature set for modeling.

In [ ]:
# Create enhanced feature pipeline
enhanced_pipeline = create_enhanced_feature_pipeline(
    include_price_trends=True,
    include_neighborhoods=True,
    include_sentiment=True,
    include_energy=True,
    include_external_data=False,  # Set to True if you have external data
    n_clusters=8
)

# Transform data using the pipeline
enhanced_features = enhanced_pipeline.fit_transform(df)

print(f"Generated {enhanced_features.shape[1]} features using the enhanced pipeline")

# Add property age features
enhanced_features = pd.concat([enhanced_features, age_features], axis=1)

# Preview the combined features
enhanced_features.head()

In [ ]:
# Create a final feature set for modeling
feature_columns = enhanced_features.columns.tolist()

# Add original numeric features that might be useful
original_numeric = ['surface_area', 'rooms', 'bathrooms']
for col in original_numeric:
    if col in df.columns:
        enhanced_features[col] = df[col]
        feature_columns.append(col)

# Create dummy variables for categorical features
if 'city' in df.columns:
    city_dummies = pd.get_dummies(df['city'], prefix='city')
    enhanced_features = pd.concat([enhanced_features, city_dummies], axis=1)
    feature_columns.extend(city_dummies.columns.tolist())

# Remove any duplicate columns
feature_columns = list(dict.fromkeys(feature_columns))

# Fill missing values (this is a simple approach - consider more sophisticated imputation)
features_clean = enhanced_features[feature_columns].fillna(enhanced_features[feature_columns].median())

print(f"Final feature set contains {features_clean.shape[1]} features")

## Feature Importance Analysis

Let's train a simple model to analyze feature importance.

In [ ]:
# Prepare data for modeling
X = features_clean
y = df['price']  # Predicting price

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate the model
y_pred = rf_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Model performance:")
print(f"  RMSE: {rmse:.2f} €")
print(f"  R²: {r2:.4f}")

In [ ]:
# Analyze feature importance
importance_df, fig = feature_importance_analysis(
    rf_model, 
    X.columns, 
    top_n=20, 
    output_path='outputs/plots/feature_importance.png',
    figsize=(12, 10)
)

print("Top 20 most important features:")
importance_df.head(20)

## Saving the Model

Let's save the trained model using the ModelRegistry from model_utils.py.

In [ ]:
# Initialize the model registry
registry = ModelRegistry(registry_path='outputs/models')

# Save the model
model_id = registry.save_model(
    model=rf_model,
    model_name='real_estate_price_prediction',
    feature_names=X.columns.tolist(),
    metrics={
        'rmse': rmse,
        'r2_score': r2
    },
    metadata={
        'description': 'Random Forest model for real estate price prediction',
        'n_features': X.shape[1],
        'n_samples': X.shape[0],
        'feature_engineering': 'enhanced'
    },
    feature_importance=dict(zip(
        X.columns, 
        rf_model.feature_importances_
    ))
)

print(f"Model saved with ID: {model_id}")

## Hyperparameter Optimization

Let's use the HyperparameterOptimizer class to fine-tune our model.

In [ ]:
# Create a pipeline for optimization
from sklearn.pipeline import Pipeline

model_pipeline = Pipeline([
    ('model', RandomForestRegressor(random_state=42))
])

# Define parameter grid
param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [None, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}

# Create optimizer
optimizer = HyperparameterOptimizer(
    pipeline=model_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

In [ ]:
# Run randomized search for faster results
results = optimizer.randomized_search(
    X_train, 
    y_train,
    n_iter=20,
    random_state=42
)

print("Best parameters:")
for param, value in results['best_params'].items():
    print(f"  {param}: {value}")

print(f"\nBest score: {-results['best_score']:.2f} RMSE")

# Save optimization results
optimizer.save_results('outputs/models/hyperparameter_optimization_results.json')

In [ ]:
# Get the optimized model
best_model = optimizer.best_estimator_

# Evaluate on test data
y_pred_optimized = best_model.predict(X_test)
rmse_optimized = np.sqrt(mean_squared_error(y_test, y_pred_optimized))
r2_optimized = r2_score(y_test, y_pred_optimized)

print(f"Optimized model performance:")
print(f"  RMSE: {rmse_optimized:.2f} €")
print(f"  R²: {r2_optimized:.4f}")

# Compare with original model
print(f"\nImprovement:")
print(f"  RMSE improvement: {rmse - rmse_optimized:.2f} € ({(rmse - rmse_optimized) / rmse * 100:.2f}%)")
print(f"  R² improvement: {r2_optimized - r2:.4f} ({(r2_optimized - r2) / r2 * 100:.2f}%)")

In [ ]:
# Save the optimized model
optimized_model_id = registry.save_model(
    model=best_model,
    model_name='real_estate_price_prediction_optimized',
    feature_names=X.columns.tolist(),
    metrics={
        'rmse': rmse_optimized,
        'r2_score': r2_optimized
    },
    metadata={
        'description': 'Optimized Random Forest model for real estate price prediction',
        'n_features': X.shape[1],
        'n_samples': X.shape[0],
        'feature_engineering': 'enhanced',
        'optimization': 'randomized_search'
    },
    feature_importance=dict(zip(
        X.columns, 
        best_model.named_steps['model'].feature_importances_
    ))
)

print(f"Optimized model saved with ID: {optimized_model_id}")

## Conclusion

In this notebook, we've demonstrated the enhanced feature engineering capabilities for real estate data. We've:

1. Extracted basic features using existing transformers
2. Applied price trend analysis to capture temporal patterns
3. Created neighborhood clusters based on location and price
4. Performed sentiment analysis on property descriptions
5. Extracted energy efficiency features
6. Calculated property age features
7. Combined all features into a comprehensive dataset
8. Trained and optimized a price prediction model
9. Analyzed feature importance
10. Saved models using the model registry

These advanced feature engineering techniques can significantly improve the performance of real estate analytics models, providing deeper insights into the factors that drive property values.